In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.stats import spearmanr, pearsonr
import pandas as pd

# --------------------------------------------------
# Configuration
# --------------------------------------------------
# Path to scored ACAI file
DATA_PATH = "../data/ACAI-US79-scored.csv"

# Weighting schemes to compare
# "indicators" is treated as the baseline scheme
SCHEMES = [
    "indicators",      # baseline
    "equal",
    "policy_heavy",
    "teaching_heavy",
]

BASELINE = "indicators"

# --------------------------------------------------
# Load data
# --------------------------------------------------
df = pd.read_csv(DATA_PATH)

# --------------------------------------------------
# Add rank and percentile-rank columns
# --------------------------------------------------
def add_ranks(df, schemes):
    """
    For each weighting scheme, add:
      - rank_{scheme}: ordinal rank (1 = highest ACAI)
      - prank_{scheme}: percentile rank (0–1), more stable for comparisons
    """
    for s in schemes:
        col = f"ACAI_{s}"

        # Raw rank (descending: higher ACAI = better rank)
        df[f"rank_{s}"] = df[col].rank(
            ascending=False, method="average"
        )

        # Percentile rank (preferred for robustness analysis)
        df[f"prank_{s}"] = df[col].rank(
            ascending=False, method="average", pct=True
        )

    return df

# --------------------------------------------------
# Compute rank correlations across weighting schemes
# --------------------------------------------------
def rank_correlations(df, schemes, use_percentile=True):
    """
    Compute pairwise Spearman and Pearson correlations
    across weighting schemes using either:
      - percentile ranks (default, recommended), or
      - raw ranks
    """
    rows = []

    # Choose rank type
    prefix = "prank_" if use_percentile else "rank_"

    # Pairwise comparisons of weighting schemes
    for s1, s2 in combinations(schemes, 2):
        x = df[f"{prefix}{s1}"]
        y = df[f"{prefix}{s2}"]

        # Drop missing values (should be rare, but safe)
        mask = x.notna() & y.notna()

        rows.append({
            "scheme_1": s1,
            "scheme_2": s2,
            "spearman": spearmanr(x[mask], y[mask])[0],
            "pearson": pearsonr(x[mask], y[mask])[0],
        })

    return pd.DataFrame(rows)

# --------------------------------------------------
# Plot rank–rank scatterplots vs baseline scheme
# --------------------------------------------------
def plot_rank_scatter(df, baseline, schemes, use_percentile=True):
    """
    For each alternative weighting scheme, plot
    baseline percentile rank vs alternative percentile rank.
    """
    prefix = "prank_" if use_percentile else "rank_"

    for s in schemes:
        if s == baseline:
            continue

        plt.figure()
        plt.scatter(
            df[f"{prefix}{baseline}"],
            df[f"{prefix}{s}"]
        )

        # 45-degree reference line (perfect agreement)
        plt.plot([0, 1], [0, 1], linestyle="--")

        plt.xlabel(f"{baseline} percentile rank")
        plt.ylabel(f"{s} percentile rank")
        plt.title(f"{baseline} vs {s}")
        plt.tight_layout()
        plt.show()

# --------------------------------------------------
# Summarize rank robustness statistics
# --------------------------------------------------
def summarize_rank_robustness(df, schemes):
    """
    Summarize maximum absolute rank change (from LOO analysis)
    across weighting schemes.
    """
    cols = [f"max_abs_rank_change_{s}" for s in schemes]
    return df[cols].describe().loc[["mean", "50%", "75%", "max"]].round(2)

# --------------------------------------------------
# Run analysis
# --------------------------------------------------
df = add_ranks(df, SCHEMES)

# Rank correlations across weighting schemes
df_rank_corr = rank_correlations(df, SCHEMES, use_percentile=True).round(2)
display(df_rank_corr)

# Save correlation table
df_rank_corr.to_csv(
    "../data/ACAI-percentile-rank-correlations.csv",
    index=False
)

# Visual diagnostics
# plot_rank_scatter(df, BASELINE, SCHEMES)

# Rank robustness summary
display(summarize_rank_robustness(df, SCHEMES))


,scheme_1,scheme_2,spearman,pearson
0,indicators,equal,0.98,0.98
1,indicators,policy_heavy,0.93,0.93
2,indicators,teaching_heavy,0.99,0.99
3,equal,policy_heavy,0.97,0.97
4,equal,teaching_heavy,0.98,0.98
5,policy_heavy,teaching_heavy,0.93,0.93


,max_abs_rank_change_indicators,max_abs_rank_change_equal,max_abs_rank_change_policy_heavy,max_abs_rank_change_teaching_heavy
mean,16.28,17.37,16.96,16.54
50%,14.50,17.00,15.50,13.50
75%,22.50,23.50,22.50,21.75
max,46.00,46.00,43.50,45.00


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/ACAI-US79-scored.csv")


from scipy.stats import pearsonr, spearmanr

def loo_correlations(df, base_col, drop_cols):
    rows = []
    base = df[base_col]

    for drop_col in drop_cols:
        dropped = df[drop_col]

        valid = base.notna() & dropped.notna()

        pearson = pearsonr(base[valid], dropped[valid])[0]
        spearman = spearmanr(base[valid], dropped[valid])[0]
        mean_abs_diff = np.mean(np.abs(base[valid] - dropped[valid]))
        max_abs_diff = np.max(np.abs(base[valid] - dropped[valid]))

        rows.append({
            "base_col": base_col,
            "drop_col": drop_col,
            "pearson_r": round(pearson, 3),
            "spearman_rho": round(spearman, 3),
            "mean_abs_diff": round(mean_abs_diff, 2),
            "max_abs_diff": round(max_abs_diff, 2),
        })

    return pd.DataFrame(rows)


equal_results = loo_correlations(
    df,
    base_col="ACAI_equal",
    drop_cols=[
        "ACAI_equal_drop_k1",
        "ACAI_equal_drop_k2",
        "ACAI_equal_drop_k3",
    ]
)

equal_results


,base_col,drop_col,pearson_r,spearman_rho,mean_abs_diff,max_abs_diff
0,ACAI_equal,ACAI_equal_drop_k1,0.854,0.850,6.03,22.05
1,ACAI_equal,ACAI_equal_drop_k2,0.832,0.746,6.53,21.01
2,ACAI_equal,ACAI_equal_drop_k3,0.871,0.870,5.43,19.44


In [ ]:
# --------------------------------------------------
# Inter-annotator agreement diagnostics
# --------------------------------------------------
def extract_interannotator_agreement(df):
    """
    Extract per-university inter-annotator agreement metrics
    and return both:
      - a per-university table
      - a summary statistics table
    """
    agreement_cols = [
        "Index",
        "alpha_overall",
        "pairwise_overall",
        "alpha_domain_aggregated",
    ]

    df_agreement = df[agreement_cols].copy()

    # Summary statistics for reporting (mean, median, IQR, max)
    summary = (
        df_agreement
        .drop(columns="Index")
        .describe()
        .loc[["mean", "50%", "75%", "max"]]
    )

    return df_agreement.round(2), summary.round(2)


# --------------------------------------------------
# Inter-annotator agreement analysis
# --------------------------------------------------
df_agreement, agreement_summary = extract_interannotator_agreement(df)

# Display per-university agreement (useful for appendix / diagnostics)
# display(df_agreement.head())

# Display summary statistics (what you report in the paper)
display(agreement_summary)

# Save agreement tables
df_agreement.to_csv(
    "../data/ACAI-interannotator-agreement-by-university.csv",
    index=False
)

agreement_summary.to_csv(
    "../data/ACAI-interannotator-agreement-summary.csv"
)



,alpha_overall,pairwise_overall,alpha_domain_aggregated
mean,0.24,0.49,0.30
50%,0.26,0.48,0.30
75%,0.41,0.58,0.56
max,0.72,0.73,0.93


In [ ]:
pd.set_option("display.max_rows", None)

# --------------------------------------------------
# Ranked universities with metadata (robust version)
# --------------------------------------------------

# ---- Parse Size string (e.g., "~80k students") to numeric ----
def parse_size(size_str):
    if pd.isna(size_str):
        return np.nan

    s = size_str.lower()
    s = s.replace("students", "").replace("~", "").strip()

    if "k" in s:
        return float(s.replace("k", "").strip()) * 1000

    return float(s.replace(",", ""))

df["size_numeric"] = df["Size"].apply(parse_size)

# ---- Bucket size (tertiles: small / medium / large) ----
df["size_bucket"], bins = pd.qcut(
    df["size_numeric"],
    q=3,
    labels=["small", "medium", "large"],
    retbins=True
)
labels = ["small", "medium", "large"]
for i, label in enumerate(labels):
    print(f"{label}: {bins[i]:.0f}, {bins[i+1]:.0f}")

# ---- ACAI ----
RANK_SCHEME = "ACAI_indicators"

metadata_cols = [
    "Index",
    "Institution",
    "Type",
    "Research Activity",
    "Regional Coverage",
    "size_bucket",
    RANK_SCHEME,
]

ranked_df = (
    df[metadata_cols]
    .dropna(subset=[RANK_SCHEME])
    .sort_values(RANK_SCHEME, ascending=False)
    .reset_index(drop=True)
)

ranked_df["Rank"] = ranked_df.index + 1
ranked_df["Percentile_Rank"] = (
    ranked_df[RANK_SCHEME].rank(ascending=False, pct=True).mul(100).round(0).astype(int)
)
ranked_df[RANK_SCHEME] = ranked_df[RANK_SCHEME].round(2)
ranked_df = ranked_df[["Rank", "Percentile_Rank", "Institution", "Index", "Type", "Research Activity", "Regional Coverage", "size_bucket", "ACAI_indicators"]]

ranked_df.to_csv(
    "../data/ACAI_ranked_universities.csv",
    index=False
)

small: 1000, 8000
medium: 8000, 20000
large: 20000, 80000


,Rank,Percentile_Rank,Institution,Index,Type,Research Activity,Regional Coverage,size_bucket,ACAI_indicators
0,1,1,University of New Hampshire,U27,Public Research-Oriented,R1,Northeast,medium,81.82
1,2,3,Portland State University,U16,Public Research-Oriented,R2,West,large,80.30
2,3,3,Stanford University,U42,Private Research-Oriented,R1,West,medium,80.30
3,4,5,University of Texas at Austin,U2,Public Research-Oriented,R1,South,large,77.27
4,5,6,University of Notre Dame,U49,Private Research-Oriented,R1,Midwest,medium,75.76
5,6,8,Baylor University,U35,Private Research-Oriented,R1,South,large,74.24
6,7,8,University at Buffalo,U28,Public Research-Oriented,R1,Northeast,large,74.24
7,8,11,University of Florida,U5,Public Research-Oriented,R1,South,large,71.21
8,9,11,University of Michigan at Ann Arbor,U18,Public Research-Oriented,R1,Midwest,large,71.21
9,10,13,Rowan University,U31,Public Research-Oriented,R2,Northeast,large,71.21
